# 🧠 LLM Security Helper - Part 2: GenAI / Agentic Spec Analysis

In [ ]:
!pip install bitsandbytes flask flask-cors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from flask import Flask, request, jsonify
from flask_cors import CORS
from datetime import datetime

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')



In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
print("🚀 Loading LLaMA 3.2 3B model from Hugging Face...")
model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN
)

print("✅ Model loaded successfully!")


🚀 Loading LLaMA 3.2 3B model from Hugging Face...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model loaded successfully!


In [ ]:
OWASP_LLM_TOP_10 = {
    "LLM01": {
        "name": "Prompt Injection",
        "description": "Manipulating LLM via crafted inputs to execute unintended actions"
    },
    "LLM02": {
        "name": "Insecure Output Handling",
        "description": "Insufficient validation/sanitization of LLM outputs before downstream use"
    },
    "LLM03": {
        "name": "Training Data Poisoning",
        "description": "Tampering with training data to introduce vulnerabilities or biases"
    },
    "LLM04": {
        "name": "Model Denial of Service",
        "description": "Resource exhaustion attacks causing service degradation"
    },
    "LLM05": {
        "name": "Supply Chain Vulnerabilities",
        "description": "Compromised components, datasets, or pre-trained models"
    },
    "LLM06": {
        "name": "Sensitive Information Disclosure",
        "description": "Unintended revelation of confidential data through LLM responses"
    },
    "LLM07": {
        "name": "Insecure Plugin Design",
        "description": "Vulnerabilities in LLM plugins/extensions with inadequate access control"
    },
    "LLM08": {
        "name": "Excessive Agency",
        "description": "LLM systems granted too much autonomy leading to unintended actions"
    },
    "LLM09": {
        "name": "Overreliance",
        "description": "Excessive dependence on LLMs without oversight causing misinformation"
    },
    "LLM10": {
        "name": "Model Theft",
        "description": "Unauthorized access, copying, or exfiltration of proprietary LLM models"
    }
}


In [ ]:
# ============================================================================
# MITRE ATLAS FRAMEWORK
# ============================================================================

ATLAS_TACTICS = {
    "Reconnaissance": "Gathering information about the ML system",
    "Resource Development": "Establishing resources to support operations",
    "Initial Access": "Gaining initial access to the ML system",
    "ML Model Access": "Gaining access to the machine learning model",
    "Execution": "Running malicious code",
    "Persistence": "Maintaining foothold in the system",
    "Privilege Escalation": "Gaining higher-level permissions",
    "Defense Evasion": "Avoiding detection",
    "Credential Access": "Stealing credentials",
    "Discovery": "Learning about the system environment",
    "Collection": "Gathering data of interest",
    "ML Attack Staging": "Preparing for attacks on ML models",
    "Exfiltration": "Stealing data from the network",
    "Impact": "Disrupting availability or integrity"
}

## PART 1

In [ ]:
def analyze_code_security(code_snippet):
    """
    Analyze code for security vulnerabilities using LLaMA 3.2 3B.

    Args:
        code_snippet (str): The code to analyze

    Returns:
        dict: Analysis results with vulnerabilities and fixes
    """

    # Construct the security analysis prompt
    system_prompt = """You are an expert security analyst specializing in code security.
Analyze the provided code for security vulnerabilities with a focus on:

1. SQL Injection
2. Cross-Site Scripting (XSS)
3. Cross-Site Request Forgery (CSRF)
4. Insecure Deserialization
5. Authentication/Authorization issues
6. Command Injection
7. Path Traversal
8. Hardcoded secrets
9. Insecure cryptography
10. Buffer overflows

For each vulnerability found, provide:
- Vulnerability name and severity (Critical/High/Medium/Low)
- Specific line(s) affected
- Detailed explanation of the risk
- Recommended fix with code example

Return your analysis in JSON format with this structure:
{
  "summary": "Brief overview of findings",
  "vulnerabilities": [
    {
      "name": "Vulnerability name",
      "severity": "Critical/High/Medium/Low",
      "lines": "Line numbers or 'multiple'",
      "description": "Detailed explanation of the vulnerability",
      "risk": "Explanation of potential impact",
      "fix": "Recommended solution",
      "fixed_code": "Code example showing the fix"
    }
  ],
  "security_score": "Score out of 10 (10 = most secure)"
}

Return ONLY raw JSON. No markdown. Escape all quotes inside strings.

Focus ONLY on security issues, not code quality or refactoring suggestions."""

    user_prompt = f"""Analyze this code for security vulnerabilities:

```
{code_snippet}
```

Provide a comprehensive security analysis in JSON format."""

    try:
        # Prepare the messages in chat format
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        # Apply chat template
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize input
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=2048,
                temperature=0.3,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the response
        response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        # print("RAW LLM RESPONSE:\n", response_text)


        # Try to extract JSON from the response
        # Look for JSON content between curly braces
        import re
        import json

        # Remove markdown code fences
        cleaned = response_text.replace("```", "").strip()

        # Extract JSON object
        json_match = re.search(r'\{.*\}', cleaned, re.DOTALL)

        if json_match:
            json_str = json_match.group(0)

            # Normalize newlines inside strings
            try:
    # First attempt: parse as-is
                analysis = json.loads(json_str)
            except json.JSONDecodeError:
                # Second attempt: escape newlines
                try:
                    safe_json = json_str.replace("\n", "\\n")
                    analysis = json.loads(safe_json)
                except json.JSONDecodeError:
                    analysis = {
                        "summary": "LLM returned malformed JSON",
                        "vulnerabilities": [{
                            "name": "Parsing Error",
                            "severity": "High",
                            "lines": "N/A",
                            "description": "Model returned invalid JSON format.",
                            "risk": "Security findings may be incomplete.",
                            "fix": "Improve prompt or enforce JSON mode.",
                            "fixed_code": ""
                        }],
                        "security_score": "N/A"
                    }

        else:
            # If no JSON found
            analysis = {
                "summary": "No JSON detected in model output",
                "vulnerabilities": [{
                    "name": "Unstructured Output",
                    "severity": "Info",
                    "lines": "N/A",
                    "description": cleaned[:500],
                    "risk": "Results may be unreliable.",
                    "fix": "Force JSON-only output.",
                    "fixed_code": ""
                }],
                "security_score": "N/A"
            }

        return {
            "success": True,
            "analysis": analysis,
            "model": "LLaMA 3.2 3B Instruct",
            "raw_response": cleaned[:200] + "..." if len(cleaned) > 200 else cleaned
        }


    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }


## PART 2

In [ ]:
def analyze_genai_specs(app_specs):

    # Construct the analysis prompt
    system_prompt = f"""You are an expert AI security analyst specializing in GenAI and LLM application security.

Analyze the provided GenAI/Agentic application specifications for potential security vulnerabilities.

Use these frameworks:

1. OWASP TOP 10 FOR LLM APPLICATIONS:
{json.dumps(OWASP_LLM_TOP_10, indent=2)}

2. MITRE ATLAS TACTICS:
{json.dumps(ATLAS_TACTICS, indent=2)}

For each vulnerability identified:
- Map to specific OWASP LLM risks (LLM01-LLM10)
- Map to relevant ATLAS tactics
- Provide severity rating (Critical/High/Medium/Low)
- Explain the specific risk in context of this application
- Provide clear, actionable mitigation strategies
- Give concrete implementation examples

Return analysis in JSON format:
{{
  "application_summary": "Brief summary of the application",
  "overall_risk_level": "Critical/High/Medium/Low",
  "vulnerabilities": [
    {{
      "title": "Clear vulnerability title",
      "owasp_mapping": ["LLM01", "LLM02"],
      "atlas_mapping": ["Tactic1", "Tactic2"],
      "severity": "Critical/High/Medium/Low",
      "description": "Detailed explanation of the vulnerability",
      "attack_scenario": "Realistic attack scenario",
      "impact": "Potential consequences",
      "mitigation": "Clear mitigation strategies",
      "implementation_example": "Concrete code/config example"
    }}
  ],
  "recommendations": [
    "Prioritized security recommendations"
  ]
}}

Be SPECIFIC to the application described. Provide ACTIONABLE advice, not generic statements."""

    user_prompt = f"""Analyze this GenAI/Agentic application for security vulnerabilities:

APPLICATION SPECIFICATIONS:
{app_specs}

IMPORTANT: Return COMPLETE JSON with ALL vulnerabilities found. Do not truncate.
Provide comprehensive security analysis mapping to OWASP Top 10 for LLM Apps and MITRE ATLAS."""

    try:
        # Prepare the messages in chat format
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        # Apply chat template
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize input
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        # Generate response
        with torch.no_grad():
          outputs = model.generate(
              **inputs,
              max_new_tokens=4096,  # ✅ Increase from 3072 to 4096
              temperature=0.5,      # ✅ Increase from 0.4 for better JSON
              do_sample=True,
              top_p=0.95,           # ✅ Add top_p for better quality
              pad_token_id=tokenizer.eos_token_id
          )

        # Decode the response
        response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        # Try to extract JSON from the response
        import re
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)

        if json_match:
            json_str = json_match.group(0)
            analysis = json.loads(json_str)
        else:
            # If no JSON found, create structured response from text
            analysis = {
                "application_summary": "Analysis completed",
                "overall_risk_level": "Medium",
                "vulnerabilities": [{
                    "title": "Security Analysis Result",
                    "owasp_mapping": ["LLM01"],
                    "atlas_mapping": ["Initial Access"],
                    "severity": "Medium",
                    "description": response_text[:500],
                    "attack_scenario": "See description for details",
                    "impact": "Potential security risks identified",
                    "mitigation": "Review the full analysis",
                    "implementation_example": "N/A"
                }],
                "recommendations": ["Review security analysis carefully"]
            }

        return {
            "success": True,
            "analysis": analysis,
            "model": "LLaMA 3.2 3B Instruct",
            "frameworks_used": ["OWASP Top 10 for LLM Apps", "MITRE ATLAS"],
            "raw_response": response_text[:200] + "..." if len(response_text) > 200 else response_text
        }

    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

In [ ]:
# ============================================================================
# TEST EXAMPLES
# ============================================================================

# Example 1: Customer Support Chatbot
example_chatbot_specs = """
GenAI Customer Support Chatbot:

Features:
- Web-based chat interface for customer inquiries
- Integrates with GPT-4 for natural language responses
- Accesses customer database (name, email, order history, payment methods)
- Can process refunds up to $500 automatically
- Stores full conversation history in cloud database
- Allows customers to upload receipts/documents for claims
- Uses RAG (Retrieval Augmented Generation) with company knowledge base
- Sends automated emails based on conversation outcomes
- No human oversight for responses under $100 value

Technical Stack:
- React frontend
- Node.js backend API
- PostgreSQL database
- OpenAI API integration
- AWS S3 for document storage
"""

# Example 2: Code Generation Agent
example_code_agent_specs = """
AI Code Generation Agent:

Features:
- Autonomous code generation based on natural language requirements
- Direct access to GitHub repositories (read/write)
- Can execute generated code in sandboxed environment for testing
- Integrates with company's internal APIs and databases
- Learns from previous code generation sessions
- Can install npm/pip packages as needed
- Automated pull request creation and merging
- Slack integration for notifications

Permissions:
- Full repository access across organization
- Can create/delete branches
- CI/CD pipeline integration
- Access to production environment variables for testing
"""

# Example 3: Document Analysis System
example_doc_system_specs = """
Intelligent Document Analysis System:

Features:
- Processes uploaded PDFs, Word docs, images
- Extracts sensitive information (SSN, credit cards, PII)
- Uses LLM to summarize and categorize documents
- Stores extracted data in searchable database
- Generates automated reports
- API for third-party integrations
- Multi-tenant architecture
- OCR for scanned documents

Data Handling:
- Uploads stored in cloud storage
- Processed data cached for 90 days
- LLM queries logged for improvement
- Shared model fine-tuned on customer documents
"""

In [ ]:
# # ============================================================================
# # RUN ANALYSIS EXAMPLES
# # ============================================================================

# print("=" * 80)
# print("LLM SECURITY HELPER - PART 2: GENAI SPEC ANALYSIS")
# print("=" * 80)

# # Test with Customer Support Chatbot
# print("\n🔍 Analyzing Customer Support Chatbot...\n")
# result1 = analyze_genai_specs(example_chatbot_specs)

# if result1["success"]:
#     analysis = result1["analysis"]
#     print(f"APPLICATION: {analysis.get('application_summary', 'N/A')}")
#     print(f"\nOVERALL RISK LEVEL: {analysis.get('overall_risk_level', 'N/A')}")
#     print(f"\n🚨 Found {len(analysis.get('vulnerabilities', []))} vulnerabilities:\n")

#     for i, vuln in enumerate(analysis.get('vulnerabilities', []), 1):
#         print(f"{i}. {vuln.get('title', 'Unknown')} [{vuln.get('severity', 'Unknown')}]")
#         print(f"   OWASP: {', '.join(vuln.get('owasp_mapping', []))}")
#         print(f"   ATLAS: {', '.join(vuln.get('atlas_mapping', []))}")
#         print(f"   Description: {vuln.get('description', 'N/A')}")
#         print(f"   Impact: {vuln.get('impact', 'N/A')}")
#         print(f"   Mitigation: {vuln.get('mitigation', 'N/A')}")
#         print()

#     print("\n📋 RECOMMENDATIONS:")
#     for i, rec in enumerate(analysis.get('recommendations', []), 1):
#         print(f"{i}. {rec}")
# else:
#     print(f"❌ Error: {result1.get('error', 'Unknown error')}")

In [ ]:
!pip install pyngrok

In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok


# NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")

from google.colab import userdata

NGROK_TOKEN = userdata.get('NGROK_TOKEN')

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

In [ ]:
# app = Flask(__name__)
# CORS(app)


# @app.route('/analyze', methods=['POST'])
# def unified_analyze():
#     try:
#         data = request.get_json()
#         analysis_type = data.get("type")

#         # -------------------------
#         # PART 1 : CODE ANALYSIS
#         # -------------------------
#         if analysis_type == "code_analysis":
#             code = data.get("code", "")
#             if not code:
#                 return jsonify({"error": "No code provided"}), 400

#             # 🔁 Replace this with your real Part 1 function
#             result = analyze_code_security(code)

#             # result = {
#             #     "message": "Code analysis placeholder",
#             #     "status": "Integrate analyze_code_security() here"
#             # }

#             return jsonify(result)

#         # -------------------------
#         # PART 2 : SPEC ANALYSIS
#         # -------------------------
#         elif analysis_type == "spec_analysis":
#             specs = data.get("specs", "")
#             if not specs:
#                 return jsonify({"error": "No specifications provided"}), 400

#             result = analyze_genai_specs(specs)
#             return jsonify(result)

#         # -------------------------
#         # INVALID TYPE
#         # -------------------------
#         else:
#             return jsonify({"error": "Invalid analysis type"}), 400

#     except Exception as e:
#         return jsonify({"error": str(e)}), 500


# # -------------------------------------------------------------------
# # HEALTH CHECK
# # -------------------------------------------------------------------

# @app.route('/health', methods=['GET'])
# def health_check():
#     return jsonify({
#         "status": "healthy",
#         "service": "LLM Security Helper (Combined)",
#         "frameworks": ["OWASP Top 10 for LLM Apps", "MITRE ATLAS"]
#     })


# # -------------------------------------------------------------------
# # FRAMEWORK DATA
# # -------------------------------------------------------------------

# @app.route('/frameworks', methods=['GET'])
# def get_frameworks():
#     return jsonify({
#         "owasp_llm_top_10": OWASP_LLM_TOP_10,
#         "atlas_tactics": ATLAS_TACTICS
#     })


# # -------------------------------------------------------------------
# # START SERVER
# # -------------------------------------------------------------------

# if __name__ == "__main__":
#     print("\n" + "=" * 80)
#     print("🚀 Flask API Server Ready")
#     print("=" * 80)
#     print("POST /analyze")
#     print("GET  /health")
#     print("GET  /frameworks")
#     print("=" * 80 + "\n")

#     app.run(
#         host="0.0.0.0",
#         port=5000,
#         debug=False,
#         use_reloader=False
#     )

from flask import Flask, request, jsonify
from flask_cors import CORS
import json
import re


app = Flask(__name__)
CORS(app)

# -------------------------------------------------
# JSON SAFETY HELPER
# -------------------------------------------------

def safe_json_parse(text):
    """
    Tries to extract JSON from LLM output.
    If parsing fails, return raw text safely.
    """
    try:
        return json.loads(text)
    except:
        pass

    try:
        match = re.search(r'\{.*\}', text, re.S)
        if match:
            return json.loads(match.group())
    except:
        pass

    return {
        "analysis": text,
        "warning": "Model returned non-JSON output"
    }

# -------------------------------------------------
# MAIN ENDPOINT
# -------------------------------------------------

@app.route('/analyze', methods=['POST'])
def unified_analyze():
    try:
        data = request.get_json(force=True)
        analysis_type = data.get("type")

        print("REQUEST:", data)

        # =========================
        # PART 1 : CODE ANALYSIS
        # =========================

        if analysis_type == "code_analysis":
            code = data.get("code", "")

            if not code:
                return jsonify({"error": "No code provided"}), 400

            raw_result = analyze_code_security(code)

            # If function already returns dict -> good
            if isinstance(raw_result, dict):
                return jsonify(raw_result)

            # If returns string -> try to parse
            parsed = safe_json_parse(raw_result)
            return jsonify(parsed)

        # =========================
        # PART 2 : SPEC ANALYSIS
        # =========================

        elif analysis_type == "spec_analysis":
            specs = data.get("specs", "")

            if not specs:
                return jsonify({"error": "No specifications provided"}), 400

            raw_result = analyze_genai_specs(specs)

            if isinstance(raw_result, dict):
                return jsonify(raw_result)

            parsed = safe_json_parse(raw_result)
            return jsonify(parsed)

        # =========================
        # INVALID TYPE
        # =========================

        else:
            return jsonify({"error": "Invalid analysis type"}), 400

    except Exception as e:
        print("BACKEND ERROR:", e)
        return jsonify({"error": str(e)}), 500

# -------------------------------------------------
# HEALTH
# -------------------------------------------------

@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({
        "status": "healthy",
        "service": "LLM Security Helper (Combined)",
        "parts": ["Code Analysis", "Spec Analysis"]
    })

# -------------------------------------------------
# START SERVER
# -------------------------------------------------

if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("🚀 Flask API Server Ready")
    print("=" * 80)
    print("POST /analyze")
    print("GET  /health")
    print("=" * 80 + "\n")

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=True,
        use_reloader=False
    )




🚀 Flask API Server Ready
POST /analyze
GET  /health

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/Feb/2026 11:33:00] "OPTIONS /analyze HTTP/1.1" 200 -


REQUEST: {'type': 'code_analysis', 'code': 'def get_user(username):\n    query = "SELECT * FROM users WHERE username = \'" + username + "\'"\n    return db.execute(query)'}


INFO:werkzeug:127.0.0.1 - - [06/Feb/2026 11:33:14] "POST /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Feb/2026 11:33:55] "OPTIONS /analyze HTTP/1.1" 200 -


REQUEST: {'type': 'code_analysis', 'code': 'from flask import Flask, request\n\n@app.route(\'/search\')\ndef search():\n    query = request.args.get(\'q\')\n    return f"<h1>Search results for: {query}</h1>"'}


INFO:werkzeug:127.0.0.1 - - [06/Feb/2026 11:34:52] "POST /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Feb/2026 11:35:35] "OPTIONS /analyze HTTP/1.1" 200 -


REQUEST: {'type': 'code_analysis', 'code': 'import os\n\ndef backup_file(filename):\n    os.system(f"tar -czf backup.tar.gz {filename}")'}


INFO:werkzeug:127.0.0.1 - - [06/Feb/2026 11:35:50] "POST /analyze HTTP/1.1" 200 -
